# Exercise 8.1: Hough transform

From *Programming in High Energy Particle Physics*, Chapter 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch08/ex08_1_solution.ipynb)

Exercise 8.1 Hough transform Implement a simple Hough transform for straight line finding in a pixel detector. Generate tracks with various slopes and intercepts, add noise hits, and evaluate the efficiency and fake rate as a function of hit multiplicity.

<details><summary>Hint</summary>

Parameterize lines as \(\rho = x\cos\theta + y\sin\theta\), so that vertical lines cause no trouble. Each hit votes for a sinusoidal curve in a 2D histogram of \((\theta, \rho)\), for example with np.histogram2d . Peaks above a threshold are track candidates. Count a track as found if a candidate has most of its hits, and a candidate as fake if it matches no true track.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)

### 1. Generate six-layer tracks and noise hits

Track matching uses the known simulated truth only for evaluation, not during Hough voting.

In [ ]:
x_planes=np.linspace(0,10,6)  # cm
truth=[(0.35,-2.0),(-0.28,2.2)]  # (slope, intercept)
def make_hits(n_noise,rng):
    xs=[];ys=[];labels=[]
    for tid,(m,b) in enumerate(truth):
        xs.extend(x_planes);ys.extend(m*x_planes+b+rng.normal(0,0.03,len(x_planes)));labels.extend([tid]*len(x_planes))
    xs.extend(rng.uniform(0,10,n_noise));ys.extend(rng.uniform(-5,5,n_noise));labels.extend([-1]*n_noise)
    return np.asarray(xs),np.asarray(ys),np.asarray(labels)
theta=np.linspace(0,np.pi,181,endpoint=False)
rho_edges=np.linspace(-11,11,221)
def hough_candidates(x,y,threshold=4):
    acc=np.zeros((len(theta),len(rho_edges)-1),dtype=int)
    for xi,yi in zip(x,y):
        rho=xi*np.cos(theta)+yi*np.sin(theta)
        ri=np.searchsorted(rho_edges,rho,side="right")-1
        valid=(ri>=0)&(ri<acc.shape[1])
        np.add.at(acc,(np.arange(len(theta))[valid],ri[valid]),1)
    cells=np.argwhere(acc>=threshold)
    cells=sorted(cells,key=lambda ij:acc[tuple(ij)],reverse=True)
    chosen=[];supports=[]
    for ti,ri in cells:
        r=0.5*(rho_edges[ri]+rho_edges[ri+1])
        support=set(np.flatnonzero(np.abs(x*np.cos(theta[ti])+y*np.sin(theta[ti])-r)<0.16))
        if len(support)<threshold: continue
        if any(len(support&s)/max(1,len(support|s))>0.5 for s in supports): continue
        chosen.append((theta[ti],r));supports.append(support)
    return acc,chosen,supports
x,y,labels=make_hits(30,rng)
acc,candidates,supports=hough_candidates(x,y)
print("Candidates:",len(candidates))
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].scatter(x,y,c=np.where(labels<0,2,labels),s=20,cmap="tab10")
axs[0].set(xlabel="x [cm]",ylabel="y [cm]",title="Pixel hits")
axs[1].imshow(acc.T,origin="lower",aspect="auto",extent=[0,np.pi,rho_edges[0],rho_edges[-1]])
axs[1].set(xlabel="θ [rad]",ylabel="ρ [cm]",title="Hough votes")
plt.tight_layout();plt.show()

### 2. Measure efficiency and fake-candidate fraction

In [ ]:
noise_levels=[0,10,30,60,100]
trials=40
eff=[];fake=[]
for nn in noise_levels:
    n_found=n_true=0;n_fake=n_candidate=0
    for _ in range(trials):
        x,y,labels=make_hits(nn,rng)
        _,cand,supports=hough_candidates(x,y)
        matched=set()
        for support in supports:
            hits=np.array(sorted(support),dtype=int)
            counts=[np.count_nonzero(labels[hits]==tid) for tid in range(len(truth))]
            best=int(np.argmax(counts))
            if counts[best]>=4 and counts[best]/len(support)>=0.8:matched.add(best)
            else:n_fake+=1
        n_found+=len(matched);n_true+=len(truth);n_candidate+=len(cand)
    eff.append(n_found/n_true);fake.append(n_fake/max(n_candidate,1))
fig,ax=plt.subplots()
ax.plot(noise_levels,eff,"o-",label="track efficiency")
ax.plot(noise_levels,fake,"s-",label="fake candidate fraction")
ax.set(xlabel="Noise hits / event",ylabel="Fraction",ylim=(0,1.05));ax.legend();plt.show()
print("efficiency:",eff,"fake fraction:",fake)
assert all(0<=v<=1 for v in eff+fake)

At low noise the two six-hit lines give strong Hough peaks. Additional random hits create accidental peaks, raising the fake-candidate fraction; the measured curves depend on binning, threshold, and matching tolerance. This is a straight-line toy detector, without magnetic bending or detector inefficiency.